In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [2]:
cols = ["flength","fwidth","fsize","fconc","fconc1","fasym","fm3long","fm3trans","falpha","fdisk","class"]
df = pd.read_csv("magic04.data",names=cols)
df.head()


,flength,fwidth,fsize,fconc,fconc1,fasym,fm3long,fm3trans,falpha,fdisk,class
0,28.7967,16.0021,2.6449,0.3918,0.1982,27.7004,22.0110,-8.2027,40.0920,81.8828,g
1,31.6036,11.7235,2.5185,0.5303,0.3773,26.2722,23.8238,-9.9574,6.3609,205.2610,g
2,162.0520,136.0310,4.0612,0.0374,0.0187,116.7410,-64.8580,-45.2160,76.9600,256.7880,g
3,23.8172,9.5728,2.3385,0.6147,0.3922,27.2107,-6.4633,-7.1513,10.4490,116.7370,g
4,75.1362,30.9205,3.1611,0.3168,0.1832,-5.5277,28.5525,21.8393,4.6480,356.4620,g


In [3]:
df["class"] = (df["class"] == "g").astype(int)

In [4]:
df = df.sample(frac=1).reset_index(drop=True)

train = df[:int(0.6*len(df))]
valid = df[int(0.6*len(df)):int(0.8*len(df))]
test  = df[int(0.8*len(df)):]

In [5]:
def scale_dataset(dataframe):
    x = dataframe[dataframe.columns[:-1]].values
    y = dataframe[dataframe.columns[-1]].values

    scaler = StandardScaler()
    x = scaler.fit_transform(x)

    data = np.hstack((x,np.reshape(y,(-1,1))))

    return data,x,y

print(len(train[train["class"]==0]))
print(len(train[train["class"]==1]))


4006
7406


In [6]:
train, xtrain,ytrain = scale_dataset(train)
valid, xvalid,yvalid = scale_dataset(valid)
test, xtest,ytest = scale_dataset(test)

In [7]:
from sklearn.neighbors import KNeighborsClassifier



In [8]:
knnmodel = KNeighborsClassifier(n_neighbors=1)
knnmodel.fit(xtrain,ytrain)


,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",1
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [9]:
ypred = knnmodel.predict(xtest)


In [10]:
from sklearn.metrics import classification_report

print(classification_report(ytest,ypred))




              precision    recall  f1-score   support

           0       0.78      0.67      0.72      1343
           1       0.83      0.90      0.86      2461

    accuracy                           0.82      3804
   macro avg       0.81      0.78      0.79      3804
weighted avg       0.82      0.82      0.81      3804



In [11]:
from sklearn.naive_bayes import  GaussianNB

nbmodel = GaussianNB()
nbmodel = nbmodel.fit(xtrain,ytrain)

ypred = nbmodel.predict(xtest)

print(classification_report(ytest,ypred))


              precision    recall  f1-score   support

           0       0.73      0.39      0.51      1343
           1       0.73      0.92      0.82      2461

    accuracy                           0.73      3804
   macro avg       0.73      0.65      0.66      3804
weighted avg       0.73      0.73      0.71      3804



In [14]:
from sklearn.linear_model import LogisticRegression

lg = LogisticRegression()
lg = lg.fit(xtrain,ytrain)

print(classification_report(ytest,lg.predict(xtest)))


              precision    recall  f1-score   support

           0       0.78      0.60      0.68      1343
           1       0.81      0.91      0.85      2461

    accuracy                           0.80      3804
   macro avg       0.79      0.75      0.76      3804
weighted avg       0.79      0.80      0.79      3804



In [15]:
from sklearn.svm import  SVC

sv = SVC()
sv = sv.fit(xtrain,ytrain)
print(classification_report(ytest,sv.predict(xtest)))

              precision    recall  f1-score   support

           0       0.90      0.69      0.78      1343
           1       0.85      0.96      0.90      2461

    accuracy                           0.86      3804
   macro avg       0.88      0.82      0.84      3804
weighted avg       0.87      0.86      0.86      3804

